# SemEval 2026 Task 11 - Bedrock Pipeline
## Syllogistic Reasoning with AWS Bedrock

**Architecture**: Uses AWS Bedrock instead of HuggingFace models

**Pipeline Components**:
1. `Config` - Configuration management
2. `DataLoader` - Data loading and preprocessing
3. `BedrockModelManager` - Bedrock client management
4. `PromptBuilder` - Prompt construction
5. `BedrockInferenceEngine` - Bedrock inference
6. `PrologExecutor` - SWI-Prolog program execution
7. `Evaluator` - Metrics calculation
8. `SemEvalPipelineBedrock` - Main orchestrator

## 1. Install Dependencies

Run this once, then comment it out.

In [1]:
!pip install -q -U boto3 pyswip pandas matplotlib tqdm

## 2. Load AWS Credentials

**Option A: Load from external file (recommended)**

Credentials are stored in `aws_credentials.txt` file.

In [2]:
import sys
from pathlib import Path

# Add this folder and lib/ to path (so load_credentials and pipeline classes work)
_nb_dir = Path.cwd() / "pamaldi_semeval_2026_11_task1" if (Path.cwd() / "pamaldi_semeval_2026_11_task1").exists() else Path.cwd()
sys.path.insert(0, str(_nb_dir))
import setup_path

from load_credentials import load_credentials_from_file, verify_credentials

# Load credentials from file (this folder or parent)
_creds = _nb_dir / "aws_credentials.txt" if (_nb_dir / "aws_credentials.txt").exists() else _nb_dir.parent / "aws_credentials.txt"
credentials = load_credentials_from_file(str(_creds))
print(f'✓ Loaded {len(credentials)} credentials from file')

# Verify credentials are set
verify_credentials()

✓ Loaded 2 credentials from file
✓ All required credentials are set
  Region: us-east-1
  Token: ABSKQmVkcm9ja0FQSUtl...


True

**Option B: Set manually (alternative)**

Uncomment and run this cell if you prefer to set credentials directly in the notebook.

In [3]:
import os
# 
# # Set AWS credentials manually
# os.environ['AWS_BEARER_TOKEN_BEDROCK'] = 'your_token_here'
# os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
# 
# print('✓ AWS credentials set')

In [4]:
import os

# Typical install path — adjust if you installed elsewhere
os.environ['SWI_HOME_DIR'] = r'C:\Program Files\swipl'

# You may also need to add the bin folder to PATH
os.environ['PATH'] = r'C:\Program Files\swipl\bin' + os.pathsep + os.environ['PATH']

from pyswip import Prolog
prolog = Prolog()

## 3. Import Classes

All classes are defined in `semeval_pipeline_classes_bedrock.py`

In [5]:
from semeval_pipeline_classes_bedrock import (
    Config,
    DataLoader,
    BedrockModelManager,
    PromptBuilder,
    BedrockInferenceEngine,
    PrologExecutor,
    CotExecutor,
    Evaluator,
    SemEvalPipelineBedrock
)

# Import Reflexion module for self-correcting execution
from reflexion_module_bedrock import ReflexionExecutorBedrock

print('✓ All classes imported successfully')
print('✓ Reflexion module imported')


✓ All classes imported successfully
✓ Reflexion module imported


## 4. Configure Pipeline

Configure the pipeline with Bedrock settings.

In [6]:
config = Config(
    save_dir="./semeval_models",
    results_dir="./semeval_results",
    model_id="qwen.qwen3-32b-v1:0",  # Bedrock model ID
    region_name="us-east-1",
    num_training=1,  # Change this to control training examples
    num_test=194,  # 0 = skip test data
    batch_size=10,
    max_new_tokens=512,
    temperature=0.7,
    top_k=20
)

config.create_directories()
print(config.to_dict())

✓ Directories created
{'save_dir': './semeval_models', 'results_dir': './semeval_results', 'model_id': 'qwen.qwen3-32b-v1:0', 'region_name': 'us-east-1', 'bearer_token': None, 'num_training': 1, 'num_test': 194, 'batch_size': 10, 'max_new_tokens': 512, 'temperature': 0.7, 'top_k': 20, 'train_data_url': 'https://raw.githubusercontent.com/neuro-symbolic-ai/semeval_2026_task_11/refs/heads/main/train_data/subtask%201/train_data.json', 'test_data_url': 'https://raw.githubusercontent.com/neuro-symbolic-ai/semeval_2026_task_11/refs/heads/main/test_data/subtask%201/test_data_subtask_1.json'}


## 5. Load Data

In [7]:
data_loader = DataLoader(config)

# Load training data
train_data = data_loader.get_train_subset()
print(f"Training examples: {len(train_data)}")

# Load test data
test_data = data_loader.get_test_subset()
print(f"Test examples: {len(test_data)}")

✓ Data exists at ./train_data/train_data.json
✓ Loaded 960 training examples
Training examples: 1
✓ Data exists at ./test_data/test_data_subtask_1.json
✓ Loaded 191 test examples
Test examples: 191


## 6. Initialize Bedrock Client

In [8]:
model_manager = BedrockModelManager(config)
client = model_manager.get_client()

print(f"\n✓ Bedrock client initialized")
print(f"Model: {config.model_id}")
print(f"Region: {config.region_name}")


Initializing Bedrock client...
Model: qwen.qwen3-32b-v1:0
Region: us-east-1
✓ Bedrock client initialized

✓ Bedrock client initialized
Model: qwen.qwen3-32b-v1:0
Region: us-east-1


## 7. Build Prompts

In [9]:
prompt_builder = PromptBuilder()
prompt_builder.use_prolog_prompt()  # Use SWI-Prolog prompt

print("✓ Using Prolog prompt")
print(f"System prompt length: {len(prompt_builder.system_prompt)} characters")

✓ Using Prolog prompt
System prompt length: 9031 characters


## 8. Run Inference

In [10]:
inference_engine = BedrockInferenceEngine(config, model_manager)

# Run on training data
train_results = inference_engine.run_inference(train_data, prompt_builder)

print(f"\nExample result:")
print(f"ID: {train_results[0]['id']}")
print(f"Response: {train_results[0]['response'][:200]}...")


Running Bedrock inference on 1 examples...


Inference: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]

✓ Inference complete

Example result:
ID: 50146f21-d265-4e3a-8d93-8165cdbe89a3
Response: ```prolog
% Counterexample: an animal that is a vehicle (e.g., a horse)
animal(horse1).
vehicle(horse1).

% A-premise: All cars are vehicles
vehicle(X) :- car(X).

% E-premise: No animal is a car
conf...


## 9. Execute Programs

In [11]:
from reflexion_module_bedrock import ReflexionMode

# Create base executor (for code extraction/execution)
base_executor = PrologExecutor(config, prompt_builder)

# Create Reflexion executor (self-correcting)
reflexion_executor = ReflexionExecutorBedrock(
    config=config,
    model_manager=model_manager,
    prompt_builder=prompt_builder,
    inference_engine=inference_engine,
    base_executor=base_executor,
    reflexion_mode=ReflexionMode.LABEL_FREE,
    max_attempts=3,  # Try up to 3 times per syllogism
    use_ground_truth=True,  # Use ground truth to determine success
    run_name='train'  # Folder will be: reflexion_train_YYYYMMDD_HHMMSS
)

# Process with reflexion (self-correcting)
print("\nProcessing with Reflexion (self-correcting execution)...")
predictions = reflexion_executor.process_results(train_results)

print(f"\nExample prediction:")
print(predictions[0])

# View reflexion statistics
import json

stats_file = os.path.join(reflexion_executor.execution_dir, 'statistics.json')
with open(stats_file, 'r') as f:
    stats = json.load(f)

print("\n" + "="*60)
print("REFLEXION STATISTICS")
print("="*60)
print(f"Total syllogisms: {stats['total_syllogisms']}")

# Handle None values (when no ground truth available)
if stats['success_rate'] is not None:
    print(f"Success rate: {stats['success_rate']*100:.1f}%")
else:
    print(f"Success rate: N/A (no ground truth)")

if stats['first_attempt_success_rate'] is not None:
    print(f"First attempt success: {stats['first_attempt_success']} ({stats['first_attempt_success_rate']*100:.1f}%)")
else:
    print(f"First attempt success: N/A")

print(f"Reflexion helped: {stats['reflexion_helped']}")
if stats['reflexion_improvement_rate'] is not None and stats['reflexion_improvement_rate'] > 0:
    print(f"Reflexion improvement: {stats['reflexion_improvement_rate']*100:.1f}%")

print("\nAttempts distribution:")
for attempts, count in sorted(stats['attempts_distribution'].items()):
    print(f"  {attempts} attempt(s): {count}")

if stats['failure_types']:
    print("\nFailure types encountered:")
    for failure_type, count in stats['failure_types'].items():
        print(f"  {failure_type}: {count}")

print(f"\n✓ Detailed logs saved to: {reflexion_executor.execution_dir}")
print(f"  - config.json: Run configuration")
print(f"  - statistics.json: Aggregate statistics")
print(f"  - summary.txt: Human-readable summary")
print(f"  - logs/: Individual attempt logs (JSON + TXT)")
print(f"  - programs/: Generated programs for each attempt")

# IMPORTANT: Update executor variable for evaluation
executor = reflexion_executor

✓ System prompt saved to: ./semeval_results\execution_prolog_20260301_132853\system_prompt.txt
✓ Execution directory: ./semeval_results\execution_prolog_20260301_132853
  Loaded 10 reflexion prompts
✓ ReflexionExecutor (Bedrock) initialized
  Max attempts: 3
  Use ground truth: True
  Reflexion mode: label_free
  Execution folder: ./semeval_results\reflexion_train_20260301_132853

Processing with Reflexion (self-correcting execution)...

Processing 1 syllogisms with Reflexion...
Max attempts per syllogism: 3
Reflexion mode: label_free


Reflexion: 100%|██████████| 1/1 [00:00<00:00,  2.88it/s]

✓ Simple report saved to: ./semeval_results\reflexion_train_20260301_132853\simple_report.txt
✓ Validity/Plausibility breakdown saved to: ./semeval_results\reflexion_train_20260301_132853\validity_plausibility_breakdown.txt
✓ No failures to analyze (all predictions correct)
⚠ No failure analysis report found, skipping pattern summary

✓ Reflexion complete
✓ Logs and programs saved to: ./semeval_results\reflexion_train_20260301_132853
✓ Bedrock calls: 1 (retries: 0)

Example prediction:
{'id': '50146f21-d265-4e3a-8d93-8165cdbe89a3', 'prediction': 'INVALID', 'label': 'INVALID'}

REFLEXION STATISTICS
Total syllogisms: 1
Success rate: 100.0%
First attempt success: 1 (100.0%)
Reflexion helped: 0

Attempts distribution:
  1 attempt(s): 1

Failure types encountered:
  unknown: 1

✓ Detailed logs saved to: ./semeval_results\reflexion_train_20260301_132853
  - config.json: Run configuration
  - statistics.json: Aggregate statistics
  - summary.txt: Human-readable summary
  - logs/: Individual a

## 10. Evaluate Results

In [12]:
import sys
import json
import os
from pathlib import Path

print("="*80)
print("OFFICIAL EVALUATION (Task 1 & 3)")
print("="*80)

# Use reflexion_executor if using Reflexion, otherwise use executor
# The executor variable is set in Cell 9
reference_file = os.path.join(executor.execution_dir, 'reference.json')
predictions_file = os.path.join(executor.execution_dir, 'predictions.json')
results_file = os.path.join(executor.execution_dir, 'evaluation_results.json')

# Build reference file
print("\n[1] Building reference file...")
reference_data = []
for item in train_data:
    reference_data.append({
        'id': item['id'],
        'validity': item['validity'],
        'plausibility': item['plausibility']
    })

with open(reference_file, 'w') as f:
    json.dump(reference_data, f, indent=2)
print(f"✓ Reference file: {reference_file}")

# Build predictions file
print("\n[2] Building predictions file...")
predictions_official = []
for pred in predictions:
    validity = pred['prediction'] == 'VALID'
    predictions_official.append({
        'id': pred['id'],
        'validity': validity
    })

with open(predictions_file, 'w') as f:
    json.dump(predictions_official, f, indent=2)
print(f"✓ Predictions file: {predictions_file}")

# Run evaluation
print("\n[3] Running official evaluation...")
print("-"*80)

_eval_kit = Path.cwd().parent / "evaluation_kit" / "task 1 & 3"
if _eval_kit.exists():
    sys.path.insert(0, str(_eval_kit))
    from evaluation_script import run_full_scoring
    run_full_scoring(reference_file, predictions_file, results_file)
else:
    print("Evaluation kit not found at:", _eval_kit)

# Display results
if os.path.exists(results_file):
    print("✓ Evaluation complete!\n")
    
    with open(results_file, 'r') as f:
        eval_results = json.load(f)
    
    print("="*80)
    print("OFFICIAL EVALUATION RESULTS")
    print("="*80)
    print(f"\nModel: {config.model_id}")
    print(f"Examples: {len(train_data)}")
    print(f"\nMetrics:")
    print(f"  Accuracy:        {eval_results['accuracy']:.2f}%")
    print(f"  Content Effect:  {eval_results['content_effect']:.2f}%")
    print(f"  Combined Score:  {eval_results['combined_score']:.2f}%")
    print(f"\n" + "="*80)
    
    # Save summary
    summary_file = os.path.join(executor.execution_dir, 'evaluation_summary.txt')
    with open(summary_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("OFFICIAL EVALUATION RESULTS\n")
        f.write("="*80 + "\n\n")
        f.write(f"Model: {config.model_id}\n")
        f.write(f"Examples: {len(train_data)}\n\n")
        f.write("Metrics:\n")
        f.write(f"  Accuracy:        {eval_results['accuracy']:.2f}%\n")
        f.write(f"  Content Effect:  {eval_results['content_effect']:.2f}%\n")
        f.write(f"  Combined Score:  {eval_results['combined_score']:.2f}%\n")
        f.write("\n" + "="*80 + "\n")
    
    print(f"\n✓ Summary saved to: {summary_file}")
else:
    print("✗ Evaluation failed (results file not created).")

OFFICIAL EVALUATION (Task 1 & 3)

[1] Building reference file...
✓ Reference file: ./semeval_results\reflexion_train_20260301_132853\reference.json

[2] Building predictions file...
✓ Predictions file: ./semeval_results\reflexion_train_20260301_132853\predictions.json

[3] Running official evaluation...
--------------------------------------------------------------------------------
Scoring complete. Results written to ./semeval_results\reflexion_train_20260301_132853\evaluation_results.json
✓ Evaluation complete!

OFFICIAL EVALUATION RESULTS

Model: qwen.qwen3-32b-v1:0
Examples: 1

Metrics:
  Accuracy:        100.00%
  Content Effect:  50.00%
  Combined Score:  20.28%


✓ Summary saved to: ./semeval_results\reflexion_train_20260301_132853\evaluation_summary.txt


## 11. Test Data Evaluation (Optional)

Generate predictions for test data if needed.

In [13]:
if config.num_test == 0:
    print("⊘ Skipping test section (num_test = 0)")
else:
    print("\n" + "="*80)
    print("TEST DATA EVALUATION WITH REFLEXION")
    print("="*80)
    
    # Run inference on test data
    print(f"\nRunning inference on {len(test_data)} test examples...")
    test_results = inference_engine.run_inference(test_data, prompt_builder)
    print(f"✓ Generated {len(test_results)} responses")
    
    # Create base executor for test data
    test_base_executor = PrologExecutor(config, prompt_builder)
    
    # Create Reflexion executor for test data
    # Note: use_ground_truth=False because test data has no labels
    # Set verbose=True to see detailed processing logs
    test_reflexion_executor = ReflexionExecutorBedrock(
        config=config,
        model_manager=model_manager,
        prompt_builder=prompt_builder,
        inference_engine=inference_engine,
        base_executor=test_base_executor,
        max_attempts=3,
        use_ground_truth=False,  # No ground truth for test data
        verbose=True,  # Enable detailed console logging
        run_name='test'  # Folder will be: reflexion_test_YYYYMMDD_HHMMSS
    )
    
    print(f"\nExecuting {len(test_results)} test programs with Reflexion...")
    print("(Note: Success = valid VALID/INVALID prediction, not correctness)")
    
    # Process and execute test programs
    test_predictions = test_reflexion_executor.process_results(test_results)
    
    print(f"\n✓ Execution complete")
    print(f"\nSample prediction:")
    print(test_predictions[0])
    
    # View test statistics
    test_stats_file = os.path.join(test_reflexion_executor.execution_dir, 'statistics.json')
    with open(test_stats_file, 'r') as f:
        test_stats = json.load(f)
    
    print("\n" + "="*60)
    print("TEST REFLEXION STATISTICS")
    print("="*60)
    print(f"Total: {test_stats['total_syllogisms']}")
    
    # Handle None success_rate (no ground truth available for test data)
    if test_stats['success_rate'] is not None:
        print(f"Success rate: {test_stats['success_rate']*100:.1f}%")
    else:
        print(f"Success rate: N/A (no ground truth for test data)")
    print(f"(Success = code executed without errors)")
    
    print("\nAttempts distribution:")
    for attempts, count in sorted(test_stats['attempts_distribution'].items()):
        print(f"  {attempts} attempt(s): {count}")
    
    if test_stats['failure_types']:
        print("\nFailure types encountered:")
        for failure_type, count in test_stats['failure_types'].items():
            print(f"  {failure_type}: {count}")
    
    # Save predictions in official format
    print("\n" + "="*60)
    print("SAVING TEST PREDICTIONS")
    print("="*60)
    
    # Convert to official format
    test_predictions_official = []
    for pred in test_predictions:
        validity = pred['prediction'] == 'VALID'
        test_predictions_official.append({
            'id': pred['id'],
            'validity': validity
        })
    
    # Save predictions
    predictions_file = os.path.join(config.results_dir, "test_predictions.json")
    with open(predictions_file, 'w') as f:
        json.dump(test_predictions_official, f, indent=2)
    
    print(f"✓ Test predictions saved to: {predictions_file}")
    print(f"  Total predictions: {len(test_predictions_official)}")
    
    # Count predictions
    valid_count = sum(1 for p in test_predictions_official if p['validity'])
    invalid_count = sum(1 for p in test_predictions_official if not p['validity'])
    
    print(f"\nPrediction distribution:")
    print(f"  VALID:   {valid_count} ({valid_count/len(test_predictions_official)*100:.1f}%)")
    print(f"  INVALID: {invalid_count} ({invalid_count/len(test_predictions_official)*100:.1f}%)")
    
    print(f"\n✓ Detailed logs saved to: {test_reflexion_executor.execution_dir}")
    print("="*60)
    
    # Official evaluation on test (if test set has labels)
    if test_data and len(test_data) > 0 and 'validity' in test_data[0]:
        import sys
        from pathlib import Path
        print("\n" + "=" * 80)
        print("OFFICIAL EVALUATION — TEST (Task 1 & 3)")
        print("=" * 80)
        test_output_dir = config.results_dir
        reference_file = os.path.join(test_output_dir, 'test_reference.json')
        predictions_file = os.path.join(test_output_dir, 'predictions_official.json')
        results_file = os.path.join(test_output_dir, 'official_evaluation_results_test.json')
        reference_data = [{'id': item['id'], 'validity': item.get('validity', False), 'plausibility': item.get('plausibility', False)} for item in test_data]
        with open(reference_file, 'w', encoding='utf-8') as f:
            json.dump(reference_data, f, indent=2)
        with open(predictions_file, 'w', encoding='utf-8') as f:
            json.dump(test_predictions_official, f, indent=2)
        _eval_kit = Path.cwd().parent / "evaluation_kit" / "task 1 & 3"
        if _eval_kit.exists() and str(_eval_kit) not in sys.path:
            sys.path.insert(0, str(_eval_kit))
        from evaluation_script import run_full_scoring
        run_full_scoring(reference_file, predictions_file, results_file)
        if os.path.exists(results_file):
            with open(results_file, 'r') as f:
                eval_results = json.load(f)
            print(f"Accuracy: {eval_results.get('accuracy', 0):.2f}%  Content Effect: {eval_results.get('content_effect', 0):.2f}%  Combined: {eval_results.get('combined_score', 0):.2f}%")
        print("=" * 80)


TEST DATA EVALUATION WITH REFLEXION

Running inference on 191 test examples...

Running Bedrock inference on 191 examples...


Inference: 100%|██████████| 191/191 [04:57<00:00,  1.56s/it]


✓ Inference complete
✓ Generated 191 responses
✓ System prompt saved to: ./semeval_results\execution_prolog_20260301_133351\system_prompt.txt
✓ Execution directory: ./semeval_results\execution_prolog_20260301_133351
  Loaded 10 reflexion prompts
✓ ReflexionExecutor (Bedrock) initialized
  Max attempts: 3
  Use ground truth: False
  Reflexion mode: label_free
  Execution folder: ./semeval_results\reflexion_test_20260301_133351

Executing 191 test programs with Reflexion...
(Note: Success = valid VALID/INVALID prediction, not correctness)

Processing 191 syllogisms with Reflexion...
Max attempts per syllogism: 3
Reflexion mode: label_free
Verbose mode: ON

[1/191]
────────────────────────────────────────────────────────────
Processing: bff2af61-d4b0-4147-8...
Ground truth: INVALID
  Attempt 1/3... extracting code... ✓ (288 chars) executing... done ✓ Prediction: VALID | Type: wrong_prediction
  → Generating reflection...
    [Bedrock] Connecting to generate reflection... ✓ OK (3050 chars)